Version 3: SSA Market Screening Model
Objective

Identify Sub-Saharan African countries that warrant further investigation by a regional African company considering cross-border expansion into decentralized-energy or mini-grid markets.

In [6]:
# Import libraries
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [26]:
import pandas as pd
from src.pipeline import prepare_ssa_dataset


from src.load_data import (
    load_worldbank_data,
    load_metadata,
)



In [8]:
# Load household consumption expenditure data
consumption_df = load_worldbank_data(
    project_root
    / "data/raw/constant 2015/API_NE.CON.PRVT.PC.KD_DS2_en_csv_v2_35532.csv"
)

consumption_df.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Households and NPISHs Final consumption expend...,NE.CON.PRVT.PC.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Households and NPISHs Final consumption expend...,NE.CON.PRVT.PC.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,1005.184135,1006.383123,995.187569,958.960804,989.489654,1004.758994,1005.915583,1011.027108,1025.994803,NaN
2,Afghanistan,AFG,Households and NPISHs Final consumption expend...,NE.CON.PRVT.PC.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,Households and NPISHs Final consumption expend...,NE.CON.PRVT.PC.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,Households and NPISHs Final consumption expend...,NE.CON.PRVT.PC.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,2248.726083,2261.455722,1961.330289,2124.217605,2341.178666,2503.621988,2761.379199,2830.939417,NaN,NaN


In [9]:
consumption_df.shape

(265, 71)

In [10]:
consumption_df[["2020", "2021", "2022", "2023", "2024", "2025"]].notna().sum()

2020    211
2021    210
2022    210
2023    205
2024    201
2025    157
dtype: int64

In [11]:
ssa_df = prepare_ssa_dataset()

ssa_df.shape

(48, 84)

In [12]:
consumption_ssa = consumption_df[
    consumption_df["Country Code"].isin(ssa_df["Country Code"])
].copy()

consumption_ssa[["2020", "2021", "2022", "2023", "2024", "2025"]].notna().sum()

2020    42
2021    42
2022    42
2023    42
2024    41
2025    37
dtype: int64

In [13]:
consumption_ssa.shape[0]

48

In [14]:
consumption_ssa.loc[
    consumption_ssa["2023"].isna(),
    ["Country Name", "Country Code"]
]

,Country Name,Country Code
69,Eritrea,ERI
130,Liberia,LBR
167,Malawi,MWI
173,Nigeria,NGA
215,South Sudan,SSD
263,Zambia,ZMB


In [15]:
consumption_subset = consumption_ssa[
    ["Country Code", "2023"]
].copy()

consumption_subset.rename(
    columns={
        "2023": "Household_Consumption_Per_Capita_2023"
    },
    inplace=True
)

consumption_subset.head()

,Country Code,Household_Consumption_Per_Capita_2023
4,AGO,2761.379199
16,BDI,194.426700
18,BEN,832.281667
19,BFA,477.779337
33,BWA,3238.181468


In [16]:
consumption_ssa.loc[
    consumption_ssa["2023"].isna(),
    [
        "Country Name",
        "Country Code",
        "2019",
        "2020",
        "2021",
        "2022",
        "2023",
    ]
]

,Country Name,Country Code,2019,2020,2021,2022,2023
69,Eritrea,ERI,NaN,NaN,NaN,NaN,NaN
130,Liberia,LBR,NaN,NaN,NaN,NaN,NaN
167,Malawi,MWI,NaN,NaN,NaN,NaN,NaN
173,Nigeria,NGA,NaN,NaN,NaN,NaN,NaN
215,South Sudan,SSD,NaN,NaN,NaN,NaN,NaN
263,Zambia,ZMB,NaN,NaN,NaN,NaN,NaN


In [17]:
year_columns = [
    column for column in consumption_ssa.columns
    if column.isdigit()
]

missing_countries = consumption_ssa[
    consumption_ssa["2023"].isna()
].copy()

for _, row in missing_countries.iterrows():
    available = row[year_columns].dropna()

    if available.empty:
        print(f"{row['Country Name']}: No data")
    else:
        latest_year = available.index[-1]
        latest_value = available.iloc[-1]

        print(
            f"{row['Country Name']}: "
            f"{latest_year} — {latest_value:.2f}"
        )

Eritrea: No data
Liberia: No data
Malawi: No data
Nigeria: No data
South Sudan: No data
Zambia: 2015 — 603.81


In [18]:
# Load Households and NPISHs final consumption expenditure per capita, PPP (constant 2015 international $) data
ppp_consumption_df = load_worldbank_data(
    project_root
    / "data/raw/pcap/API_NE.CON.PCAP.PP.KD_DS2_en_csv_v2_219.csv"
)

ppp_consumption_df.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Households and NPISHs Final consumption expend...,NE.CON.PCAP.PP.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,19616.988902,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Households and NPISHs Final consumption expend...,NE.CON.PCAP.PP.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,2593.758787,2587.780073,2577.988142,2492.061725,2560.152670,2594.137724,2596.096038,2615.295977,2657.610349,NaN
2,Afghanistan,AFG,Households and NPISHs Final consumption expend...,NE.CON.PCAP.PP.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2358.701101,1730.800083,1716.297201,1787.194039,1992.356971,NaN,NaN
3,Africa Western and Central,AFW,Households and NPISHs Final consumption expend...,NE.CON.PCAP.PP.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,Households and NPISHs Final consumption expend...,NE.CON.PCAP.PP.KD,NaN,NaN,NaN,NaN,NaN,NaN,...,3662.003683,3682.733635,3193.985607,3459.244215,3812.560792,4077.096366,4496.848625,4610.125994,NaN,NaN


In [19]:
ppp_consumption_df[["2021", "2022", "2023", "2024", "2025"]].notna().sum()

2021    217
2022    210
2023    205
2024    201
2025    161
dtype: int64

In [20]:
ppp_consumption_ssa = ppp_consumption_df[
    ppp_consumption_df["Country Code"].isin(ssa_df["Country Code"])
].copy()

ppp_consumption_ssa[
    ["2021", "2022", "2023", "2024", "2025"]
].notna().sum()

2021    44
2022    43
2023    43
2024    42
2025    38
dtype: int64

In [21]:
ppp_consumption_ssa.shape[0]

48

In [22]:
ppp_consumption_ssa.loc[
    ppp_consumption_ssa["2023"].isna(),
    ["Country Name", "Country Code"]
]

,Country Name,Country Code
69,Eritrea,ERI
130,Liberia,LBR
173,Nigeria,NGA
215,South Sudan,SSD
263,Zambia,ZMB


In [23]:
# load Poverty headcount ratio at $3.00 a day (2021 PPP)
poverty_df = load_worldbank_data(
    project_root
    / "data/raw/poverty/API_SI.POV.DDAY_DS2_en_csv_v2_34547.csv"
)

poverty_df.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Poverty headcount ratio at $3.00 a day (2021 P...,SI.POV.DDAY,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Poverty headcount ratio at $3.00 a day (2021 P...,SI.POV.DDAY,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Poverty headcount ratio at $3.00 a day (2021 P...,SI.POV.DDAY,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,Poverty headcount ratio at $3.00 a day (2021 P...,SI.POV.DDAY,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,Poverty headcount ratio at $3.00 a day (2021 P...,SI.POV.DDAY,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,39.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
poverty_ssa = poverty_df[
    poverty_df["Country Code"].isin(ssa_df["Country Code"])
].copy()

poverty_ssa.shape

(48, 71)

In [27]:
year_columns = [
    column for column in poverty_ssa.columns
    if column.isdigit()
]

latest_poverty_records = []

for _, row in poverty_ssa.iterrows():
    available = row[year_columns].dropna()

    if available.empty:
        latest_poverty_records.append({
            "Country Name": row["Country Name"],
            "Country Code": row["Country Code"],
            "Latest Year": None,
            "Poverty Rate": None,
        })
    else:
        latest_poverty_records.append({
            "Country Name": row["Country Name"],
            "Country Code": row["Country Code"],
            "Latest Year": int(available.index[-1]),
            "Poverty Rate": available.iloc[-1],
        })

latest_poverty_df = pd.DataFrame(latest_poverty_records)

latest_poverty_df.sort_values(
    ["Latest Year", "Country Name"],
    na_position="first"
)

,Country Name,Country Code,Latest Year,Poverty Rate
12,Eritrea,ERI,NaN,NaN
36,"Somalia, Fed. Rep.",SOM,NaN,NaN
9,"Congo, Rep.",COG,2011.0,39.2
33,Sudan,SDN,2014.0,10.1
4,Botswana,BWA,2015.0,21.4
11,Cabo Verde,CPV,2015.0,14.6
29,Namibia,NAM,2015.0,22.9
39,Eswatini,SWZ,2016.0,44.5
15,Ghana,GHA,2016.0,39.0
21,Liberia,LBR,2016.0,33.6


In [28]:
latest_poverty_df["Poverty Rate"].notna().sum()

np.int64(46)

In [29]:
latest_poverty_df["Latest Year"].value_counts(
    dropna=False
).sort_index()

Latest Year
2011.0     1
2014.0     1
2015.0     3
2016.0     4
2017.0     4
2018.0     5
2019.0     4
2020.0     3
2021.0    12
2022.0     7
2023.0     1
2024.0     1
NaN        2
Name: count, dtype: int64

In [30]:
latest_poverty_df[
    latest_poverty_df["Latest Year"].isna()
    | (latest_poverty_df["Latest Year"] < 2018)
].sort_values("Latest Year")

,Country Name,Country Code,Latest Year,Poverty Rate
9,"Congo, Rep.",COG,2011.0,39.2
33,Sudan,SDN,2014.0,10.1
11,Cabo Verde,CPV,2015.0,14.6
4,Botswana,BWA,2015.0,21.4
29,Namibia,NAM,2015.0,22.9
37,South Sudan,SSD,2016.0,76.5
21,Liberia,LBR,2016.0,33.6
15,Ghana,GHA,2016.0,39.0
39,Eswatini,SWZ,2016.0,44.5
27,Mauritius,MUS,2017.0,0.5


Note: Excludiing the following from our analysis due to the following
- Household consumption PPP was rejected because four countries have no observations, including Nigeria, and common-year coverage was incomplete.
- Poverty headcount was rejected because only 33 of 48 countries had observations from 2018 onward, while the latest observations ranged from 2011 to 2024.

In [31]:
# load industry value added per capita (constant 2015 US$) data
industry_df = load_worldbank_data(
    project_root
    / "data/raw/industry_value_added/API_NV.IND.TOTL.ZS_DS2_en_csv_v2_102950.csv"
)

industry_df.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Industry (including construction), value added...",NV.IND.TOTL.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,12.394828,11.358156,11.625867,12.525487,10.977495,9.728798,10.837942,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,"Industry (including construction), value added...",NV.IND.TOTL.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,25.404757,26.161139,26.040470,25.064852,26.456724,27.130492,26.467975,25.875000,26.222208,NaN
2,Afghanistan,AFG,"Industry (including construction), value added...",NV.IND.TOTL.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,10.051874,13.387247,14.058112,12.952600,14.273657,16.050368,13.449823,14.520351,NaN,NaN
3,Africa Western and Central,AFW,"Industry (including construction), value added...",NV.IND.TOTL.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,24.415911,26.687096,23.339454,21.227491,21.897342,23.064715,23.244644,24.032747,23.247403,NaN
4,Angola,AGO,"Industry (including construction), value added...",NV.IND.TOTL.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,36.696930,40.902353,42.170234,37.138063,38.994569,35.947321,35.701247,33.884228,27.067926,NaN


In [32]:
industry_ssa = industry_df[
    industry_df["Country Code"].isin(ssa_df["Country Code"])
].copy()

industry_ssa[
    ["2020", "2021", "2022", "2023", "2024", "2025"]
].notna().sum()

2020    45
2021    45
2022    45
2023    45
2024    45
2025    45
dtype: int64

In [33]:
industry_ssa.shape[0]

48

In [34]:
industry_ssa.loc[
    industry_ssa["2023"].isna(),
    ["Country Name", "Country Code"]
]

,Country Name,Country Code
69,Eritrea,ERI
212,"Somalia, Fed. Rep.",SOM
215,South Sudan,SSD


In [35]:
industry_ssa[
    [
        "Country Name",
        "Country Code",
        "2020",
        "2021",
        "2022",
        "2023",
        "2024",
        "2025",
    ]
].head(10)

,Country Name,Country Code,2020,2021,2022,2023,2024,2025
4,Angola,AGO,37.138063,38.994569,35.947321,35.701247,33.884228,27.067926
16,Burundi,BDI,15.096000,16.137944,16.074813,17.368597,17.262975,18.228510
18,Benin,BEN,16.290549,16.467400,17.017827,17.302198,17.383828,17.504319
19,Burkina Faso,BFA,29.469477,28.530624,26.043961,25.757414,26.512828,30.063227
33,Botswana,BWA,26.690568,33.040033,37.457130,34.066940,29.080464,27.883701
34,Central African Republic,CAF,20.448189,21.587320,21.157300,20.706817,20.033456,21.223509
41,Cote d'Ivoire,CIV,20.702203,22.103035,23.044542,23.900637,24.043860,23.928182
42,Cameroon,CMR,23.038065,24.492554,27.130359,25.130871,23.235096,24.774457
43,"Congo, Dem. Rep.",COD,35.161685,42.352938,41.957389,39.500969,38.965560,39.705477
44,"Congo, Rep.",COG,33.951760,42.893703,49.245552,45.223634,40.121557,35.612291


In [36]:
industry_subset = industry_ssa[
    ["Country Code", "Country Name", "2023"]
].copy()

industry_subset.rename(
    columns={
        "2023": "Industry_Value_Added_Percent_GDP_2023"
    },
    inplace=True
)

In [37]:
industry_subset.sort_values(
    "Industry_Value_Added_Percent_GDP_2023",
    ascending=False
).head(10)

,Country Code,Country Name,Industry_Value_Added_Percent_GDP_2023
79,GAB,Gabon,53.869796
87,GNQ,Equatorial Guinea,48.293837
44,COG,"Congo, Rep.",45.223634
43,COD,"Congo, Dem. Rep.",39.500969
4,AGO,Angola,35.701247
263,ZMB,Zambia,35.107778
223,SWZ,Eswatini,34.506439
33,BWA,Botswana,34.066940
264,ZWE,Zimbabwe,33.255552
165,MRT,Mauritania,30.936473


In [38]:
industry_subset.sort_values(
    "Industry_Value_Added_Percent_GDP_2023",
    ascending=True,
    na_position="last"
).head(10)

,Country Code,Country Name,Industry_Value_Added_Percent_GDP_2023
218,STP,Sao Tome and Principe,2.727154
46,COM,Comoros,10.329577
47,CPV,Cabo Verde,10.977669
225,SYC,Seychelles,13.295596
86,GNB,Guinea-Bissau,15.412000
85,GMB,"Gambia, The",16.422421
120,KEN,Kenya,17.122215
18,BEN,Benin,17.302198
16,BDI,Burundi,17.368597
172,NER,Niger,17.414420


In [39]:
industry_subset[
    industry_subset["Country Code"] == "NGA"
]

,Country Code,Country Name,Industry_Value_Added_Percent_GDP_2023
173,NGA,Nigeria,19.130238


In [41]:
# load electric power consumption (kWh per capita) data
electric_power_consumption_df = load_worldbank_data(
    project_root
    / "data/raw/electric_power_consump/API_EG.USE.ELEC.KH.PC_DS2_en_csv_v2_37832.csv"
)

electric_power_consumption_df.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Electric power consumption (kWh per capita),EG.USE.ELEC.KH.PC,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Electric power consumption (kWh per capita),EG.USE.ELEC.KH.PC,NaN,NaN,NaN,NaN,NaN,NaN,...,566.130764,568.138013,550.050979,513.707046,513.465287,502.713913,484.004412,NaN,NaN,NaN
2,Afghanistan,AFG,Electric power consumption (kWh per capita),EG.USE.ELEC.KH.PC,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,Electric power consumption (kWh per capita),EG.USE.ELEC.KH.PC,NaN,NaN,NaN,NaN,NaN,NaN,...,180.330748,182.329386,187.800544,195.399901,202.381652,200.328677,207.964937,NaN,NaN,NaN
4,Angola,AGO,Electric power consumption (kWh per capita),EG.USE.ELEC.KH.PC,NaN,NaN,NaN,NaN,NaN,NaN,...,315.199297,370.736573,411.266103,439.118174,394.730414,373.031828,379.103010,NaN,NaN,NaN


In [43]:
electricity_use_ssa = electric_power_consumption_df[
    electric_power_consumption_df["Country Code"].isin(ssa_df["Country Code"])
].copy()

electricity_use_ssa[
    ["2020", "2021", "2022", "2023", "2024", "2025"]
].notna().sum()

2020    32
2021    32
2022    32
2023    32
2024     1
2025     0
dtype: int64

In [45]:
electricity_use_ssa.shape[0]

48

In [46]:
electricity_use_ssa.loc[
    electricity_use_ssa["2023"].isna(),
    ["Country Name", "Country Code"]
]

,Country Name,Country Code
16,Burundi,BDI
34,Central African Republic,CAF
46,Comoros,COM
47,Cabo Verde,CPV
84,Guinea,GIN
85,"Gambia, The",GMB
86,Guinea-Bissau,GNB
130,Liberia,LBR
140,Lesotho,LSO
157,Mali,MLI


# Version 3 Close-out: Indicator Screening

## Purpose

Version 3 extended the Version 2 electricity-access and GDP-per-capita model by testing additional national-level indicators for two proposed dimensions:

1. Household economic capacity; and
2. Productive or business electricity demand.

The objective was not to add every available variable, but to retain only indicators that were both conceptually appropriate and sufficiently comparable across the 48 Sub-Saharan African countries in the model.

## Decision rule

A primary scored indicator should have a credible value for at least 90% of the 48 countries in a common recent reference year, including Nigeria unless countries are explicitly labelled as not assessed. Indicators also had to measure the intended construct without creating systematically misleading country rankings.

## Indicators assessed

| Candidate indicator | Evidence from screening | Decision |
|---|---|---|
| GDP per capita, 2023 | Broad common-year coverage; consistent cross-country comparison. | **Retained** as the primary national proxy for economic capacity. |
| Household consumption per capita, PPP (constant 2021 international $) | 43/48 countries had a 2023 value. Eritrea, Liberia, Nigeria, South Sudan and Zambia were missing; allowing Zambia's 2021 value would raise coverage only to 44/48. | **Rejected** from the primary score. |
| Poverty headcount ratio at $3.00/day (2021 PPP) | 46/48 countries had an observation, but only 33/48 were from 2018 onward. Latest observations ranged from 2011 to 2024. | **Rejected** from the primary score. |
| Industry value added (% of GDP), 2023 | 45/48 countries covered. The highest values were concentrated in extractive economies such as Gabon, Equatorial Guinea, Congo Republic and Angola, while Kenya ranked low. | **Rejected** as a productive-demand proxy. |
| Electric power consumption per capita, 2023 | 32/48 countries covered (66.7%); the series was not consistently maintained beyond 2023. | **Rejected** from the primary score. |

## Why GDP per capita remains

GDP per capita is not a direct measure of rural household affordability, willingness to pay, or productive electricity demand. However, it was retained because it provides the most consistent 2023 comparison across the full SSA screening population. The alternatives tested were either incomplete for strategically important countries, measured in non-comparable years, or did not validly represent decentralised-energy demand.

## Explicit exclusions

- Household consumption PPP was rejected because four countries have no observations, including Nigeria, and common-year coverage was incomplete.
- Poverty headcount was rejected because only 33 of 48 countries had observations from 2018 onward, while the latest observations ranged from 2011 to 2024.
- Industry value added (% of GDP) was rejected because it rewards sector composition, including extractive dependence, rather than broad-based productive electricity demand.
- Electric power consumption per capita was rejected because its 2023 coverage was only 32 of 48 countries.

## Version 3 outcome

Version 3 did **not** add a new scored economic-demand variable beyond GDP per capita. This is a deliberate model-design decision: forcing poorly matched national indicators into a composite score would make the ranking look more sophisticated without making it more credible.

The primary model therefore retains its comparable, full-sample economic-capacity measure and documents the tested alternatives and their limitations.

## Handoff to Version 4

Version 4 will move from broad macroeconomic demand proxies to **market readiness / enabling environment** for decentralised-energy and mini-grid investment.

It will distinguish:

- **Minimum feasibility conditions:** a mini-grid/off-grid regulatory pathway, a defined licensing or permit route, and a tariff-setting framework.
- **Readiness-strength factors:** grid-arrival or compensation rules, and an electrification plan that explicitly supports decentralised renewable energy.

High electricity need alone should not be treated as a near-term investment opportunity where the legal and revenue framework for mini-grid development is unclear.
